In [ ]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# === 設定資料夾路徑 ===
data_folder = Path(r"C:\Users\irene\OneDrive\桌面\AVI defect code data\Defect code data")
output_folder = data_folder / "output"
output_folder.mkdir(exist_ok=True)

# === 所有 Excel 檔案 ===
excel_files = list(data_folder.glob("*.xlsx"))

# === defect code 欄位（需根據實際使用更新） ===
defect_code_cols = ['100', '101', '102', '117', '200', '201', '203', '205', '500', '501', '502', '504', '507']

# === 欄位對應轉換表（實際欄位 → 資料庫欄位） ===
column_map = {
    'MFG Date': 'mfg_date',
    'Site': 'site',
    'Customer Id': 'customer_id',
    'Prdgrp Id': 'product_group_id',
    'Stage': 'stage',
    'Income Gross Die': 'income_gross_die',
    'Inspected Wafer Gross Die': 'inspected_wafer_gross_die',
    'Inspected Die': 'inspected_die',
    'Total Defect Die': 'total_defect_die',
    'Wafer size(inch)': 'wafer_size_inch',
    'Wafer THK(mil)': 'wafer_thk_mil',
    'Production type': 'production_type',
    'CLASSIFICATION': 'classification',
    'CLASS': 'class_label'
}

def process_excel(file_path, start_index):
    df = pd.read_excel(file_path, header=1)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns=column_map)

    record_count = df.shape[0]
    inspection_ids = list(range(start_index, start_index + record_count))

    # 主資料表
    wafer_cols = list(column_map.values())
    main_df = df[wafer_cols].copy()
    main_df.insert(0, 'inspection_id', inspection_ids)

    # defect breakdown
    breakdown_cols = [c for c in df.columns if "." in c and c not in defect_code_cols]
    breakdown_records = []
    for i, ins_id in enumerate(inspection_ids):
        for col in breakdown_cols:
            breakdown_records.append({
                'inspection_id': ins_id,
                'defect_type': col,
                'defect_count': df.iloc[i][col]
            })
    breakdown_df = pd.DataFrame(breakdown_records)

    # defect code logs
    code_records = []
    for i, ins_id in enumerate(inspection_ids):
        for col in defect_code_cols:
            if col in df.columns:
                code_records.append({
                    'inspection_id': ins_id,
                    'defect_code': col,
                    'defect_count': df.iloc[i][col]
                })
    code_df = pd.DataFrame(code_records)

    return main_df, breakdown_df, code_df, start_index + record_count

# === 整合所有檔案 ===
main_all = []
breakdown_all = []
code_all = []
skipped_files = []
current_index = 1

for file in excel_files:
    print(f"Processing: {file.name}")
    try:
        m_df, b_df, c_df, current_index = process_excel(file, current_index)
        main_all.append(m_df)
        breakdown_all.append(b_df)
        code_all.append(c_df)
    except Exception as e:
        print(f"⚠️ 跳過檔案 {file.name}：{e}")
        skipped_files.append(file.name)

# === 合併並輸出 ===
pd.concat(main_all, ignore_index=True).to_csv(output_folder / "wafer_inspections.csv", index=False)
pd.concat(breakdown_all, ignore_index=True).to_csv(output_folder / "defect_breakdown.csv", index=False)
pd.concat(code_all, ignore_index=True).to_csv(output_folder / "defect_code_logs.csv", index=False)

print("\n✅ 匯出完成：所有 CSV 已儲存到 output 資料夾")
if skipped_files:
    print("\n⚠️ 以下檔案未成功處理：")
    for f in skipped_files:
        print(f" - {f}")
